# NB10-CV: Hierarchical Bayesian Prototype — Dec/Jan Leave-One-Winter-Out Cross-Validation

Repeats the analysis from `10_hierarchical_bayesian_prototype.ipynb` using a cross-validation scheme  
where each test fold contains all Dec + Jan days from one winter season.

**Fold design:** Leave-one-winter-out  
- 5 folds: Dec2021+Jan2022, Dec2022+Jan2023, Dec2023+Jan2024, Dec2024+Jan2025, Dec2025+Jan2026  
- Training set = all data except the test fold  
- Source data (different farm) is kept fixed across all folds as prior information  

**Persistence note:** Not applicable — NB10 uses weather/calendar features only, no AR1 term.

In [1]:
import os
os.environ['PYTENSOR_FLAGS'] = 'floatX=float64'
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
import pytensor.tensor as pt
from sklearn.metrics import brier_score_loss, roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler

DATA_DIR = '/Users/dlau/repos/fish-welfare/data/'
OUT_DIR  = '/Users/dlau/repos/fish-welfare/ModelSelection/dec-jan/'
features = ['month_sin', 'doy_cos', 'night_precip_sum', 'precip_2d_sum', 'night_wind_min']
RANDOM_SEED = 42
a0 = 0.3

print(f"PyMC {pm.__version__}, ArviZ {az.__version__}")

PyMC 5.28.0, ArviZ 0.23.4


In [2]:
# Load priors
try:
    prior_df = pd.read_csv(DATA_DIR + 'nb07v2_source_posterior.csv').set_index('param')
    beta_prior_mean = prior_df.loc[features, 'pymc_mean'].values
    beta_prior_std  = prior_df.loc[features, 'pymc_std'].values / np.sqrt(a0)
    alpha_prior_mean = float(prior_df.loc['alpha', 'pymc_mean'])
    print("Loaded priors from nb07v2_source_posterior.csv")
except Exception as e:
    print(f"Falling back to defaults: {e}")
    beta_prior_mean = np.zeros(len(features))
    beta_prior_std  = np.ones(len(features)) * 2.0 / np.sqrt(a0)
    alpha_prior_mean = 0.0

print(f"Prior means: {dict(zip(features, beta_prior_mean.round(3)))}")

# Load source data (fixed across folds — different farm)
src = pd.read_csv(DATA_DIR + 'nb04_source_features.csv', parse_dates=['date'])
src_c = src[features + ['frac_low', 'n_ponds']].dropna().copy()
src_c['n_total'] = src_c['n_ponds'].astype(int)
src_c['n_low'] = np.minimum(
    (src_c['frac_low'] * src_c['n_total']).round().astype(int), src_c['n_total'])
src_c = src_c[src_c['n_total'] > 0].reset_index(drop=True)
n_src = src_c['n_total'].values.astype(int)
k_src = src_c['n_low'].values.astype(int)

# Fit scaler on all source features for the joint model
scaler_src = StandardScaler()
X_src_std = scaler_src.fit_transform(src_c[features].values)

# Load target data
tgt = pd.read_csv(DATA_DIR + 'nb04_target_features.csv', parse_dates=['date']).sort_values('date')
tgt_c = tgt[features + ['n_total', 'n_low', 'frac_low', 'date', 'bad_day']].dropna().copy()
tgt_c['n_total'] = tgt_c['n_total'].astype(int)
tgt_c['n_low']   = np.minimum(tgt_c['n_low'].astype(int), tgt_c['n_total'])
tgt_c = tgt_c[tgt_c['n_total'] > 0].reset_index(drop=True)
tgt_c['month'] = tgt_c['date'].dt.month
tgt_c['year']  = tgt_c['date'].dt.year

print(f"Source: {len(src_c)} obs, Target: {len(tgt_c)} obs")

Loaded priors from nb07v2_source_posterior.csv
Prior means: {'month_sin': np.float64(0.216), 'doy_cos': np.float64(0.282), 'night_precip_sum': np.float64(0.162), 'precip_2d_sum': np.float64(-0.112), 'night_wind_min': np.float64(-0.131)}
Source: 53 obs, Target: 746 obs


In [3]:
# Define 5 leave-one-winter-out folds
# Each fold: test = Dec_yr + Jan_(yr+1)
# 'winter_id' = year in which December falls
FOLD_DEFS = [
    {'name': 'Dec2021-Jan2022', 'dec_year': 2021, 'jan_year': 2022},
    {'name': 'Dec2022-Jan2023', 'dec_year': 2022, 'jan_year': 2023},
    {'name': 'Dec2023-Jan2024', 'dec_year': 2023, 'jan_year': 2024},
    {'name': 'Dec2024-Jan2025', 'dec_year': 2024, 'jan_year': 2025},
    {'name': 'Dec2025-Jan2026', 'dec_year': 2025, 'jan_year': 2026},
]

def is_test_day(row, fold):
    return ((row['month'] == 12 and row['year'] == fold['dec_year']) or
            (row['month'] ==  1 and row['year'] == fold['jan_year']))

for fold in FOLD_DEFS:
    mask = tgt_c.apply(lambda r: is_test_day(r, fold), axis=1)
    fold['test_idx'] = tgt_c.index[mask].tolist()
    fold['train_idx'] = tgt_c.index[~mask].tolist()
    print(f"Fold {fold['name']}: train={len(fold['train_idx'])}, test={len(fold['test_idx'])}")

Fold Dec2021-Jan2022: train=724, test=22
Fold Dec2022-Jan2023: train=725, test=21
Fold Dec2023-Jan2024: train=723, test=23
Fold Dec2024-Jan2025: train=707, test=39
Fold Dec2025-Jan2026: train=711, test=35


In [4]:
def run_nb10_fold(fold, tgt_c, X_src_std, n_src, k_src,
                   beta_prior_mean, beta_prior_std, alpha_prior_mean,
                   features, a0, seed=42):
    """Fit joint BetaBinomial model for one CV fold."""
    train = tgt_c.loc[fold['train_idx']].reset_index(drop=True)
    test  = tgt_c.loc[fold['test_idx']].reset_index(drop=True)

    # Standardize target features on training set
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(train[features].values)
    X_test_std  = scaler.transform(test[features].values)

    n_train = train['n_total'].values.astype(int)
    k_train = train['n_low'].values.astype(int)

    with pm.Model() as model:
        alpha_src = pm.Normal('alpha_src', mu=alpha_prior_mean, sigma=2)
        alpha_tgt = pm.Normal('alpha_tgt', 0, 5)
        beta = pm.Normal('beta', mu=beta_prior_mean, sigma=beta_prior_std, shape=len(features))
        kappa = pm.Lognormal('kappa', mu=np.log(2), sigma=1)

        # Source likelihood
        eta_src = alpha_src + pm.math.dot(X_src_std, beta)
        mu_src  = pm.math.sigmoid(eta_src)
        a_src   = pm.math.clip(mu_src * kappa, 0.01, 1e4)
        b_src   = pm.math.clip((1 - mu_src) * kappa, 0.01, 1e4)
        y_src   = pm.BetaBinomial('y_src', alpha=a_src, beta=b_src, n=n_src, observed=k_src)

        # Target train likelihood
        eta_tgt = alpha_tgt + pm.math.dot(X_train_std, beta)
        mu_tgt  = pm.math.sigmoid(eta_tgt)
        a_tgt   = pm.math.clip(mu_tgt * kappa, 0.01, 1e4)
        b_tgt   = pm.math.clip((1 - mu_tgt) * kappa, 0.01, 1e4)
        y_tgt   = pm.BetaBinomial('y_tgt', alpha=a_tgt, beta=b_tgt, n=n_train, observed=k_train)

        trace = pm.sample(
            1000, tune=1000, chains=4, target_accept=0.9,
            progressbar=False, random_seed=seed,
            return_inferencedata=True
        )

    # Posterior predictive on test
    beta_samp     = trace.posterior['beta'].values.reshape(-1, len(features))
    alpha_tgt_s   = trace.posterior['alpha_tgt'].values.flatten()

    eta_test      = alpha_tgt_s[:, None] + beta_samp @ X_test_std.T
    mu_test       = 1 / (1 + np.exp(-eta_test))
    y_pred        = mu_test.mean(0)

    y_true_bin  = test['bad_day'].values
    y_true_frac = test['frac_low'].values

    # Soft Brier (MSE vs observed frac_low, avoids sklearn binary constraint)
    brier = float(np.mean((y_pred - y_true_frac) ** 2))
    auc   = roc_auc_score(y_true_bin, y_pred) if len(np.unique(y_true_bin)) > 1 else np.nan
    ll    = log_loss(y_true_bin, np.clip(y_pred, 1e-6, 1-1e-6)) if len(np.unique(y_true_bin)) > 1 else np.nan

    rhat_max  = az.summary(trace)['r_hat'].max()
    ess_min   = az.summary(trace)['ess_bulk'].min()
    n_div     = int(trace.sample_stats['diverging'].sum())

    return dict(
        fold=fold['name'],
        n_train=len(train), n_test=len(test),
        brier=brier, auc=auc, log_loss=ll,
        rhat_max=rhat_max, ess_min=ess_min, divergences=n_div,
        y_pred=y_pred, y_true_frac=y_true_frac, y_true_bin=y_true_bin,
        dates=test['date'].values
    )


cv_results_nb10 = []
for fold in FOLD_DEFS:
    print(f"\n=== Fold: {fold['name']} ===")
    res = run_nb10_fold(fold, tgt_c, X_src_std, n_src, k_src,
                        beta_prior_mean, beta_prior_std, alpha_prior_mean,
                        features, a0, seed=RANDOM_SEED)
    cv_results_nb10.append(res)
    print(f"  Brier={res['brier']:.4f}  AUC={res['auc']:.4f}  "
          f"LogLoss={res['log_loss']:.4f}  R-hat_max={res['rhat_max']:.3f}  "
          f"Divergences={res['divergences']}")

Initializing NUTS using jitter+adapt_diag...



=== Fold: Dec2021-Jan2022 ===


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [alpha_src, alpha_tgt, beta, kappa]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 2 seconds.


...

  Brier=0.0290  AUC=0.3684  LogLoss=0.2040  R-hat_max=1.000  Divergences=0

=== Fold: Dec2025-Jan2026 ===


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [alpha_src, alpha_tgt, beta, kappa]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 2 seconds.


  Brier=0.0244  AUC=0.6176  LogLoss=0.1991  R-hat_max=1.000  Divergences=0


In [5]:
# Build summary table
rows = [{k: v for k, v in r.items()
         if k not in ('y_pred', 'y_true_frac', 'y_true_bin', 'dates')}
        for r in cv_results_nb10]
cv_df = pd.DataFrame(rows)
print("=== NB10 Dec/Jan CV Results ===")
print(cv_df.to_string(index=False))

# Pooled (all held-out Dec/Jan obs combined)
y_pred_all  = np.concatenate([r['y_pred']      for r in cv_results_nb10])
y_frac_all  = np.concatenate([r['y_true_frac'] for r in cv_results_nb10])
y_bin_all   = np.concatenate([r['y_true_bin']  for r in cv_results_nb10])

brier_pooled = float(np.mean((y_pred_all - y_frac_all) ** 2))
auc_pooled   = roc_auc_score(y_bin_all, y_pred_all) if len(np.unique(y_bin_all)) > 1 else np.nan
ll_pooled    = log_loss(y_bin_all, np.clip(y_pred_all, 1e-6, 1-1e-6)) if len(np.unique(y_bin_all)) > 1 else np.nan

print(f"\nPooled (all {len(y_pred_all)} Dec/Jan obs):")
print(f"  Brier={brier_pooled:.4f}  AUC={auc_pooled:.4f}  LogLoss={ll_pooled:.4f}")

# Load baselines for comparison
try:
    baselines = pd.read_csv(DATA_DIR + 'nb09_baseline_results.csv')
    marginal_brier = baselines[baselines['model'].str.contains('Marginal')]['brier'].values[0]
    print(f"\nMarginal baseline Brier: {marginal_brier:.4f}")
    beats = brier_pooled < marginal_brier
    print(f"NB10-CV pooled {'BEATS' if beats else 'does NOT beat'} marginal baseline")
except Exception as e:
    print(f"Could not load baselines: {e}")

=== NB10 Dec/Jan CV Results ===
           fold  n_train  n_test    brier      auc  log_loss  rhat_max  ess_min  divergences
Dec2021-Jan2022      724      22 0.023054 0.761905  0.227237       1.0   4739.0            0
Dec2022-Jan2023      725      21 0.071734 0.691176  0.497782       1.0   4979.0            0
Dec2023-Jan2024      723      23 0.095102 0.815789  0.460513       1.0   4655.0            0
Dec2024-Jan2025      707      39 0.029002 0.368421  0.203972       1.0   4911.0            0
Dec2025-Jan2026      711      35 0.024404 0.617647  0.199124       1.0   4620.0            0

Pooled (all 140 Dec/Jan obs):
  Brier=0.0442  AUC=0.4390  LogLoss=0.2926

Marginal baseline Brier: 0.0408
NB10-CV pooled does NOT beat marginal baseline


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

fold_names = [r['fold'] for r in cv_results_nb10]
xpos = np.arange(len(fold_names))

# Brier
ax = axes[0]
briers = [r['brier'] for r in cv_results_nb10]
ax.bar(xpos, briers, color='steelblue', alpha=0.8)
ax.axhline(brier_pooled, color='red', linestyle='--', label=f'Pooled={brier_pooled:.4f}')
try:
    ax.axhline(marginal_brier, color='orange', linestyle=':', label=f'Marginal={marginal_brier:.4f}')
except:
    pass
ax.set_xticks(xpos); ax.set_xticklabels(fold_names, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('Brier Score (lower is better)'); ax.set_title('NB10-CV: Brier per Fold')
ax.legend(fontsize=8)

# AUC
ax = axes[1]
aucs = [r['auc'] for r in cv_results_nb10]
ax.bar(xpos, aucs, color='seagreen', alpha=0.8)
ax.axhline(auc_pooled, color='red', linestyle='--', label=f'Pooled={auc_pooled:.4f}')
ax.set_xticks(xpos); ax.set_xticklabels(fold_names, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('AUC-ROC (higher is better)'); ax.set_title('NB10-CV: AUC per Fold')
ax.legend(fontsize=8)

# Log-loss
ax = axes[2]
lls = [r['log_loss'] for r in cv_results_nb10]
ax.bar(xpos, lls, color='coral', alpha=0.8)
ax.axhline(ll_pooled, color='red', linestyle='--', label=f'Pooled={ll_pooled:.4f}')
ax.set_xticks(xpos); ax.set_xticklabels(fold_names, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('Log-Loss (lower is better)'); ax.set_title('NB10-CV: Log-Loss per Fold')
ax.legend(fontsize=8)

plt.suptitle('NB10 Hierarchical BetaBinomial — Dec/Jan Leave-One-Winter-Out CV', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR + 'nb10_cv_metrics.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved nb10_cv_metrics.png")

Saved nb10_cv_metrics.png


In [7]:
# Predicted vs observed for each fold
fig, axes = plt.subplots(len(cv_results_nb10), 1, figsize=(14, 4 * len(cv_results_nb10)))

for ax, res in zip(axes, cv_results_nb10):
    xs = np.arange(len(res['dates']))
    ax.scatter(xs, res['y_true_frac'], color='red', s=30, zorder=5, label='Observed frac_low')
    ax.plot(xs, res['y_pred'], color='steelblue', lw=1.5, label='Posterior mean prediction')
    ax.axhline(0.3, color='gray', linestyle='--', alpha=0.5, label='Threshold 0.3')
    ax.set_title(f"Fold {res['fold']}  (n={res['n_test']}, Brier={res['brier']:.4f}, AUC={res['auc']:.4f})")
    ax.set_ylabel('frac_low')
    ax.set_xticks(xs)
    ax.set_xticklabels([pd.Timestamp(d).strftime('%b-%d') for d in res['dates']], rotation=45, ha='right', fontsize=6)
    ax.legend(fontsize=8)

plt.suptitle('NB10-CV: Predicted vs Observed (Dec/Jan test folds)', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR + 'nb10_cv_predictions.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved nb10_cv_predictions.png")

Saved nb10_cv_predictions.png


In [8]:
print("=== MCMC Diagnostics per Fold ===")
print(f"{'Fold':<22} {'R-hat_max':>10} {'ESS_min':>10} {'Divergences':>12}")
print("-" * 58)
for res in cv_results_nb10:
    print(f"{res['fold']:<22} {res['rhat_max']:>10.4f} {res['ess_min']:>10.0f} {res['divergences']:>12}")

=== MCMC Diagnostics per Fold ===
Fold                    R-hat_max    ESS_min  Divergences
----------------------------------------------------------
Dec2021-Jan2022            1.0000       4739            0
Dec2022-Jan2023            1.0000       4979            0
Dec2023-Jan2024            1.0000       4655            0
Dec2024-Jan2025            1.0000       4911            0
Dec2025-Jan2026            1.0000       4620            0


In [9]:
cv_df.to_csv(OUT_DIR + 'nb10_cv_results.csv', index=False)
print("Saved nb10_cv_results.csv")

# Also save pooled predictions
pred_df = pd.DataFrame({
    'fold': np.concatenate([[r['fold']] * len(r['y_pred']) for r in cv_results_nb10]),
    'date': np.concatenate([r['dates'] for r in cv_results_nb10]),
    'y_pred': y_pred_all,
    'y_true_frac': y_frac_all,
    'y_true_bin': y_bin_all,
})
pred_df.to_csv(OUT_DIR + 'nb10_cv_pooled_predictions.csv', index=False)
print("Saved nb10_cv_pooled_predictions.csv")
print()
print("=== NB10 CV FINAL SUMMARY ===")
print(f"Total Dec/Jan test observations: {len(y_pred_all)}")
print(f"Pooled Brier:    {brier_pooled:.4f}")
print(f"Pooled AUC:      {auc_pooled:.4f}")
print(f"Pooled Log-Loss: {ll_pooled:.4f}")
print(f"Mean Brier (across folds): {cv_df['brier'].mean():.4f} ± {cv_df['brier'].std():.4f}")
print(f"Mean AUC   (across folds): {cv_df['auc'].mean():.4f} ± {cv_df['auc'].std():.4f}")

Saved nb10_cv_results.csv
Saved nb10_cv_pooled_predictions.csv

=== NB10 CV FINAL SUMMARY ===
Total Dec/Jan test observations: 140
Pooled Brier:    0.0442
Pooled AUC:      0.4390
Pooled Log-Loss: 0.2926
Mean Brier (across folds): 0.0487 ± 0.0329
Mean AUC   (across folds): 0.6510 ± 0.1747
